<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/06-generalization-regularization-experiments.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Generalization, Regularization, and Experimental Practice** {#generalization-regularization-experimental-practice}

Chapter 05 explained how losses, gradients, and optimizers reduce error on observed examples. This chapter asks the harder question: **does that reduction reveal a reusable pattern, or only a better memory of the training set?** Generalization is the ability of a learned predictor to remain useful on fresh samples drawn from the deployment population. Regularization changes what solutions are reachable or preferred; experimental practice determines whether an apparent improvement is believable.

For a training sample $S=\{(x_i,y_i)\}_{i=1}^{N}$ drawn from an unknown distribution $\mathcal{P}$, empirical and population risk are

$$
\widehat R_S(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i),
\qquad
R(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{P}}[\ell(f_\theta(x),y)].
$$

Training can directly optimize $\widehat R_S$; deployment cares about $R$. Validation and test sets are finite estimates of population behavior, not interchangeable extra training data. The validation set supports decisions such as architecture, regularization strength, and stopping time. The test set is reserved for the final estimate after those decisions are frozen.

### **Generalization in Overparameterized Networks** {#generalization-overparameterized-networks}

A model is **overparameterized** when it has enough degrees of freedom to interpolate the training set, often with more trainable parameters than examples. Classical capacity arguments warn that a larger hypothesis class can fit more accidental patterns. Modern neural networks complicate this intuition: two networks can both reach zero training error yet represent very different functions between observed samples. Architecture, initialization, stochastic optimization, normalization, augmentation, margins, and parameter norms help select one interpolating solution from many.

A generic generalization statement has the form

$$
R(\theta) \lesssim \widehat R_S(\theta)
+\mathcal{C}(f_\theta,S)
+O\!\left(\sqrt{\frac{\log(1/\delta)}{N}}\right),
$$

with probability at least $1-\delta$. Here $\mathcal{C}$ is an **effective**, usually data-dependent complexity term. Different theories define it through margins, norms, stability, compression, PAC-Bayes quantities, or another restricted description of the learned function. This expression is a conceptual template rather than one plug-in theorem. Raw parameter count alone does not explain why a specific trained network transfers.

All runnable examples in this chapter use one experimental thread: the 1,797-sample **Optical Recognition of Handwritten Digits** dataset distributed by scikit-learn from the UCI repository. Each example is therefore a controlled change to the same ten-class problem rather than a disconnected toy. Images are $8\times8$ grayscale arrays with values from 0 to 16. We scale pixels to $[0,1]$, construct one fixed stratified 60/20/20 split, and never use the test labels for model selection. The small dataset is deliberate: every experiment runs on CPU while preserving real sampling, multiclass, and image-augmentation issues.

Dataset sources: [scikit-learn `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) and [UCI Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B) (CC BY 4.0, DOI `10.24432/C50P49`).

<details>
<summary><strong>PyTorch: establish the shared Digits experiment</strong></summary>

```python
import copy
import math
import random
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=606):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
targets = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(targets))

# The test set is separated first; validation is then separated from development data.
dev_idx, test_idx = train_test_split(
    all_indices, test_size=0.20, random_state=606, stratify=digits.target
)
train_idx, val_idx = train_test_split(
    dev_idx, test_size=0.25, random_state=606, stratify=digits.target[dev_idx]
)

x_train, y_train = images[train_idx], targets[train_idx]
x_val, y_val = images[val_idx], targets[val_idx]
x_test, y_test = images[test_idx], targets[test_idx]


def make_loader(x, y, batch_size=128, shuffle=False, seed=606):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle, generator=generator
    )


class DigitsMLP(nn.Module):
    def __init__(self, widths=(128, 64), dropout=0.0):
        super().__init__()
        layers, in_features = [], 64
        for width in widths:
            layers += [nn.Linear(in_features, width), nn.ReLU(), nn.Dropout(dropout)]
            in_features = width
        layers.append(nn.Linear(in_features, 10))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x.flatten(1))


@torch.no_grad()
def classification_metrics(model, x, y):
    model.eval()
    logits = model(x)
    return {
        "loss": F.cross_entropy(logits, y).item(),
        "accuracy": (logits.argmax(1) == y).float().mean().item(),
    }


def fit_model(model, epochs=30, lr=3e-3, weight_decay=0.0, seed=606,
              train_x=x_train, train_y=y_train, callback=None):
    seed_everything(seed)
    loader = make_loader(train_x, train_y, shuffle=True, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
        record = {
            "epoch": epoch,
            "train": classification_metrics(model, train_x, train_y),
            "validation": classification_metrics(model, x_val, y_val),
        }
        history.append(record)
        if callback is not None and callback(model, record):
            break
    return history


assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert x_train.shape[1:] == (1, 8, 8)
assert y_train.unique().numel() == 10
print({"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)})
```

</details>

The split, preprocessing, and evaluation functions above form the chapter's **experimental contract**. Later sections may alter one treatment, but they reuse the same data boundary and metric definitions. This makes comparisons interpretable and makes accidental leakage easier to detect.

### **Underfitting, Overfitting, and the Generalization Gap** {#underfitting-overfitting-generalization-gap}

**Underfitting** occurs when the pipeline cannot represent or optimize a useful rule: both training and validation performance are poor. **Overfitting** occurs when additional fitting improves the training set while validation performance stagnates or deteriorates. These diagnoses belong to the entire pipeline, not permanently to an architecture. A small model may underfit under one feature representation but work well after better preprocessing; a large model may generalize under strong augmentation but memorize corrupted labels without it.

For a matched metric and evaluation protocol, a loss gap can be written as

$$
G_{\mathrm{val}}=\widehat R_{\mathrm{val}}(\theta)-\widehat R_{\mathrm{train}}(\theta).
$$

A positive $G_{\mathrm{val}}$ is evidence of different behavior, not a complete causal diagnosis. During training, augmentation and dropout deliberately make training examples harder; during evaluation they are disabled. The training loss may then exceed validation loss even when nothing is wrong. Always compare model mode, preprocessing, reduction, and class composition before interpreting a gap.

Learning curves separate three common cases. High train and validation loss suggests representation or optimization failure. Low train loss with much higher validation loss suggests excessive fitting or a distribution mismatch. Both losses falling together suggests that additional optimization remains useful. Task metrics should accompany loss because a small cross-entropy change may primarily reflect confidence calibration rather than a change in predicted classes.

<details>
<summary><strong>PyTorch: diagnose capacity and time on the shared split</strong></summary>

```python
seed_everything(607)
linear_model = DigitsMLP(widths=())
linear_history = fit_model(linear_model, epochs=20, lr=5e-3, seed=607)

seed_everything(607)
wide_model = DigitsMLP(widths=(256, 256))
wide_history = fit_model(wide_model, epochs=70, lr=3e-3, seed=607)

checkpoints = [0, 4, 14, 29, 69]
wide_curve = [
    (
        wide_history[i]["epoch"] + 1,
        round(wide_history[i]["train"]["loss"], 4),
        round(wide_history[i]["validation"]["loss"], 4),
    )
    for i in checkpoints
]

assert wide_history[-1]["train"]["loss"] < linear_history[-1]["train"]["loss"]
assert all(math.isfinite(value) for row in wide_curve for value in row[1:])
print("linear:", linear_history[-1])
print("wide learning curve (epoch, train loss, validation loss):", wide_curve)
```

</details>

![Polynomial models illustrate underfitting, an appropriate fit, and overfitting as capacity increases.](assets/dl06-underfitting-overfitting.png){fig-align="center" width="76%" fig-alt="Three polynomial regression plots comparing underfitting, an appropriate fit, and overfitting as model capacity increases."}

*Image source: [scikit-learn, Underfitting vs. Overfitting](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html), BSD-3-Clause documentation example.*

The diagram uses polynomial regression because the shape of the fitted function is visible. The Digits experiment tests the same principle in classification: inspect trajectories rather than naming a model “overfit” from parameter count alone.

### **Explicit and Implicit Regularization** {#explicit-implicit-regularization}

Regularization is any mechanism that changes learning so that fitted performance is sacrificed, constrained, or biased in favor of behavior expected to transfer. It is broader than adding a penalty to the loss.

**Explicit regularization** appears directly in the objective or training transformation. Examples include norm penalties, label smoothing, MixUp, data augmentation, and constraints on parameters. If

$$
J(\theta)=\widehat R_S(\theta)+\lambda\Omega(\theta),
$$

$\Omega$ measures an undesirable property and $\lambda$ controls its strength. A larger $\lambda$ does not universally mean better generalization: it moves the estimator along a bias-variance trade-off and can eventually cause underfitting.

**Implicit regularization** arises from the parameterization and optimization path even when the written objective contains no penalty. Minibatch noise, initialization scale, convolutional weight sharing, optimizer choice, finite training time, and early stopping can make some solutions much easier to reach than others. For example, gradient descent on an underdetermined linear system initialized at zero converges to a minimum-Euclidean-norm interpolating solution. Deep nonlinear networks have no equally universal rule, but the principle remains: the optimizer is a solution-selection mechanism, not merely a loss minimizer.

The distinction is useful experimentally. If weight decay and augmentation are changed together, one cannot tell whether improvement came from norm control or invariance assumptions. A strong study changes one lever at a time, records the resulting train/validation behavior, and reports interactions in a later factorial comparison.

<details>
<summary><strong>PyTorch: compare two interpolating solutions under an explicit constraint</strong></summary>

```python
# Use a deterministic subset of real Digits pixels to create an underdetermined
# linear regression problem: 40 examples constrain 64 pixel coefficients.
design = x_train[:40].flatten(1)
binary_target = (y_train[:40] >= 5).float().unsqueeze(1)

minimum_norm = torch.linalg.pinv(design) @ binary_target
ridge_strength = 0.1
ridge = torch.linalg.solve(
    design.T @ design + ridge_strength * torch.eye(design.shape[1]),
    design.T @ binary_target,
)

minimum_norm_fit = F.mse_loss(design @ minimum_norm, binary_target)
ridge_fit = F.mse_loss(design @ ridge, binary_target)

# Ridge accepts more training error to shrink the coefficient vector.
assert ridge.norm() < minimum_norm.norm()
assert ridge_fit >= minimum_norm_fit - 1e-7
print({
    "minimum_norm": (minimum_norm_fit.item(), minimum_norm.norm().item()),
    "ridge": (ridge_fit.item(), ridge.norm().item()),
})
```

</details>

This small calculation isolates selection among many fitted coefficient vectors. Neural regularization is more complex, but the diagnostic question is the same: **which property of the selected solution changed, and why should that property transfer?**

### **Weight Decay and Norm Control** {#weight-decay-norm-control}

An $L_2$ penalty adds $\frac{\lambda}{2}\lVert\theta\rVert_2^2$ to the loss. Its gradient contributes $\lambda\theta$, so plain SGD updates

$$
\theta_{t+1}
=\theta_t-\eta_t\left(g_t+\lambda\theta_t\right)
=(1-\eta_t\lambda)\theta_t-\eta_tg_t.
$$

With SGD, this is algebraically equivalent to multiplicative weight decay. With adaptive optimization, inserting $\lambda\theta$ into the gradient also passes it through coordinate-wise moment normalization. AdamW instead decouples shrinkage from the adaptive gradient step:

$$
\theta_{t+1}=(1-\eta_t\lambda)\theta_t
-\eta_t\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.
$$

Decoupling makes the intended shrinkage easier to interpret, but the effective decay still depends on the learning-rate schedule and number of updates. Biases and normalization scale/shift parameters are commonly excluded because shrinking them does not impose the same function-level preference as shrinking weight matrices. Parameter grouping is therefore part of the method.

Norm control is not a guarantee that the function is simple. ReLU networks permit compensating rescalings across adjacent layers, and a small parameter norm can coexist with a sharp or poorly calibrated decision boundary. Record validation metrics and relevant norms; do not optimize a norm as though it were the deployment objective.

<details>
<summary><strong>PyTorch: measure AdamW shrinkage on Digits</strong></summary>

```python
decay_results = {}
for decay in (0.0, 1e-3, 1e-2):
    seed_everything(608)
    model = DigitsMLP(widths=(128, 64))
    history = fit_model(model, epochs=30, lr=3e-3, weight_decay=decay, seed=608)
    matrix_norm = torch.sqrt(sum(
        parameter.detach().pow(2).sum()
        for name, parameter in model.named_parameters()
        if parameter.ndim > 1
    )).item()
    decay_results[decay] = {
        "weight_norm": matrix_norm,
        "validation": history[-1]["validation"],
    }

assert decay_results[1e-2]["weight_norm"] < decay_results[0.0]["weight_norm"]
assert all(0.0 <= result["validation"]["accuracy"] <= 1.0 for result in decay_results.values())
print(decay_results)
```

</details>

The strongest decay should reduce the measured matrix norm under this fixed protocol, but validation accuracy need not improve monotonically. The scientifically meaningful output is the curve across plausible strengths, not the claim that “more regularization is better.”

### **Dropout and Stochastic Depth** {#dropout-stochastic-depth}

Dropout randomly removes activations during training. For activation vector $h$, independent mask $m_j\sim\mathrm{Bernoulli}(q)$ and keep probability $q=1-p$ produce **inverted dropout**

$$
\widetilde h=\frac{m\odot h}{q},
\qquad
\mathbb{E}[\widetilde h]=h.
$$

The $1/q$ factor keeps the expected activation unchanged, allowing evaluation to use the unmasked network directly. Dropout discourages units from relying on one exact co-adaptation pattern, but its effect depends on location. Applying high dropout to a small model or after severe information bottlenecks can destroy signal. A model must also switch correctly between `train()` and `eval()`; otherwise evaluation remains random and systematically mis-scaled behavior can appear.

**Stochastic depth** applies the same idea to residual branches. For block $h_{l+1}=h_l+F_l(h_l)$, training may use

$$
h_{l+1}=h_l+\frac{b_l}{q_l}F_l(h_l),
\qquad b_l\sim\mathrm{Bernoulli}(q_l).
$$

The identity path survives even when the residual branch is dropped, so very deep networks train over an ensemble of effective depths. Survival probabilities often decrease with depth because later blocks are more redundant. This method is structurally suited to residual networks; it is not a drop-in replacement for arbitrary hidden-unit dropout.

<details>
<summary><strong>PyTorch: verify stochastic semantics on real digit activations</strong></summary>

```python
class ResidualMLPBlock(nn.Module):
    def __init__(self, width=64, drop_path=0.0):
        super().__init__()
        self.branch = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))
        self.drop_path = drop_path

    def forward(self, x):
        residual = self.branch(x)
        if self.training and self.drop_path > 0:
            keep = 1.0 - self.drop_path
            mask = torch.bernoulli(torch.full((x.shape[0], 1), keep, device=x.device))
            residual = residual * mask / keep
        return x + residual


digit_vectors = x_train[:32].flatten(1)
dropout = nn.Dropout(p=0.5)
dropout.train()
samples = torch.stack([dropout(digit_vectors) for _ in range(200)])
empirical_mean = samples.mean(0)

block = ResidualMLPBlock(drop_path=0.5)
block.train()
block_outputs = torch.stack([block(digit_vectors) for _ in range(100)])
block.eval()
deterministic_a = block(digit_vectors)
deterministic_b = block(digit_vectors)

assert (empirical_mean - digit_vectors).abs().mean() < 0.05
assert torch.allclose(deterministic_a, deterministic_b)
assert block_outputs.var(0).mean() > 0
print("dropout expectation error:", (empirical_mean - digit_vectors).abs().mean().item())
```

</details>

Dropout changes activation-level co-adaptation; stochastic depth changes path-level computation. Compare them at matched training budgets and report evaluation-mode results. Training-mode variance is expected behavior, not uncertainty about the final deterministic network.

### **Data Augmentation** {#data-augmentation}

Data augmentation encodes a task assumption through transformations $T\sim\mathcal{A}$. Training minimizes

$$
\frac{1}{N}\sum_{i=1}^{N}
\mathbb{E}_{T\sim\mathcal{A}}
\left[\ell(f_\theta(T(x_i)),y_i)\right].
$$

This encourages invariance to transformations that preserve the label. The difficult phrase is **label preserving**. Small translations may preserve a handwritten digit, while a large rotation can turn a recognisable symbol into an ambiguous one. In medical imaging, a left-right flip may reverse anatomy; in remote sensing it may be harmless. Augmentation is therefore a model of permissible variation, not free data.

Augmentation strength should be inspected in input space and evaluated by slices. If transformed samples look unrealistic, a validation improvement can still hide degradation on clinically or operationally important subgroups. For Digits, the $8\times8$ resolution makes interpolation particularly destructive, so this chapter uses only small affine shifts and explicitly clamps the result.

<details>
<summary><strong>PyTorch: apply and verify a small translation on Digits</strong></summary>

```python
def translate_digits(batch, max_pixels=1, seed=610):
    generator = torch.Generator().manual_seed(seed)
    n = batch.shape[0]
    shifts = torch.randint(-max_pixels, max_pixels + 1, (n, 2), generator=generator)
    theta = torch.eye(2, 3).unsqueeze(0).repeat(n, 1, 1)
    # affine_grid translations are normalized to [-1, 1].
    theta[:, 0, 2] = 2.0 * shifts[:, 1] / batch.shape[-1]
    theta[:, 1, 2] = 2.0 * shifts[:, 0] / batch.shape[-2]
    grid = F.affine_grid(theta, batch.size(), align_corners=False)
    return F.grid_sample(batch, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


original = x_train[:16]
shifted = translate_digits(original)
assert shifted.shape == original.shape
assert shifted.min() >= 0 and shifted.max() <= 1
assert not torch.allclose(shifted, original)

seed_everything(610)
augmented_model = DigitsMLP(widths=(128, 64))
augmented_x = torch.cat([x_train, translate_digits(x_train, seed=611)])
augmented_y = torch.cat([y_train, y_train])
augmented_history = fit_model(
    augmented_model, epochs=25, lr=3e-3, seed=610,
    train_x=augmented_x, train_y=augmented_y,
)
print(augmented_history[-1]["validation"])
```

</details>

A stronger experiment would sample a new transform every epoch rather than materializing one copy. The compact implementation makes the information boundary visible: augment only training data, and never create related copies on opposite sides of a split.

### **MixUp, CutMix, and Label Smoothing** {#mixup-cutmix-label-smoothing}

These methods soften supervision in different ways.

For MixUp, choose two examples and $\lambda\sim\mathrm{Beta}(\alpha,\alpha)$:

$$
\widetilde x=\lambda x_i+(1-\lambda)x_j,
\qquad
\widetilde y=\lambda y_i+(1-\lambda)y_j.
$$

The target is a class-probability vector, so cross-entropy becomes $-\sum_k\widetilde y_k\log p_k$. MixUp encourages approximately linear predictions between training samples and reduces extreme confidence. It is less plausible when interpolation has no semantic meaning.

CutMix replaces a spatial region of one image with a region from another. The target weight is the **retained area fraction**, not merely the sampled scalar, because clipping the rectangle at image boundaries changes its area. CutMix preserves local texture more strongly than pixel interpolation, although the $8\times8$ Digits images expose its limitation: a small patch can remove a large semantic component.

Label smoothing replaces one-hot target $q$ with

$$
q'_k=(1-\varepsilon)q_k+\frac{\varepsilon}{K},
$$

where $K$ is the class count. It discourages infinite logit separation but does not create input invariance. MixUp/CutMix already create soft targets, so blindly stacking strong label smoothing can over-regularize and obscure which mechanism helped.

<details>
<summary><strong>PyTorch: construct all three targets from one Digits batch</strong></summary>

```python
seed_everything(611)
xb, yb = next(iter(make_loader(x_train, y_train, batch_size=32, shuffle=True, seed=611)))
permutation = torch.randperm(len(xb))
lam = 0.7

one_hot = F.one_hot(yb, num_classes=10).float()
paired = F.one_hot(yb[permutation], num_classes=10).float()
mixup_x = lam * xb + (1 - lam) * xb[permutation]
mixup_y = lam * one_hot + (1 - lam) * paired

cutmix_x = xb.clone()
top, left, height, width = 2, 2, 4, 4
cutmix_x[:, :, top:top + height, left:left + width] = xb[permutation, :, top:top + height, left:left + width]
replaced_fraction = (height * width) / (xb.shape[-2] * xb.shape[-1])
cutmix_y = (1 - replaced_fraction) * one_hot + replaced_fraction * paired

epsilon = 0.1
smoothed_y = (1 - epsilon) * one_hot + epsilon / 10
logits = DigitsMLP(widths=(64,))(mixup_x)
soft_cross_entropy = -(mixup_y * logits.log_softmax(-1)).sum(-1).mean()

assert mixup_x.shape == cutmix_x.shape == xb.shape
assert torch.allclose(mixup_y.sum(1), torch.ones(len(xb)))
assert torch.allclose(cutmix_y.sum(1), torch.ones(len(xb)))
assert torch.allclose(smoothed_y.sum(1), torch.ones(len(xb)))
assert torch.isfinite(soft_cross_entropy)
print({"mixup loss": soft_cross_entropy.item(), "CutMix replaced fraction": replaced_fraction})
```

</details>

The comparison is conceptual: augmentation changes the input distribution, MixUp and CutMix couple input composition to a soft target, and label smoothing changes only target confidence. Their usefulness depends on whether those assumptions match the task.

### **Early Stopping and Model Selection** {#early-stopping-model-selection}

Early stopping treats optimization time as a regularization parameter. Save a checkpoint whenever a declared validation metric improves, stop after a fixed **patience** without improvement, and restore the best checkpoint rather than the final one. In simple linear problems, stopping gradient descent early suppresses low-signal directions before they are fully fitted; in deep networks the same spectral story is not exact, but finite training still restricts the reachable solution.

Three details prevent common mistakes:

1. The monitored metric and its direction must be explicit. Accuracy is maximized; loss is minimized.
2. Patience is measured in evaluation events, not inherently epochs. Changing evaluation frequency changes the rule.
3. The optimizer and scheduler state belong to a resumable checkpoint, even if only model weights are needed for final inference.

Validation is no longer an unbiased evaluation set once it guides stopping. Repeatedly peeking at it, changing patience, and rerunning seeds gradually overfits validation decisions. A locked test set remains necessary.

<details>
<summary><strong>PyTorch: stop and restore on the fixed validation set</strong></summary>

```python
seed_everything(612)
early_model = DigitsMLP(widths=(256, 128), dropout=0.1)
best = {"loss": float("inf"), "state": None, "epoch": None}
patience, stale = 8, 0


def early_stopping_callback(model, record):
    global stale
    validation_loss = record["validation"]["loss"]
    if validation_loss < best["loss"] - 1e-4:
        best.update(loss=validation_loss, state=copy.deepcopy(model.state_dict()), epoch=record["epoch"])
        stale = 0
    else:
        stale += 1
    return stale >= patience


early_history = fit_model(
    early_model, epochs=100, lr=3e-3, weight_decay=1e-3,
    seed=612, callback=early_stopping_callback,
)
last_validation = classification_metrics(early_model, x_val, y_val)
early_model.load_state_dict(best["state"])
restored_validation = classification_metrics(early_model, x_val, y_val)

assert best["state"] is not None
assert restored_validation["loss"] <= last_validation["loss"] + 1e-7
print({"stopped_after": len(early_history), "best_epoch": best["epoch"] + 1,
       "restored_validation": restored_validation})
```

</details>

The code uses validation loss because it is smoother than accuracy on a 360-sample validation set. A production study would also save the optimizer, scheduler, random-number states, preprocessing configuration, and data version.

### **Double Descent and Modern Generalization Phenomena** {#double-descent-modern-generalization}

The classical bias-variance picture suggests a U-shaped test-error curve as capacity increases. **Double descent** adds a second descent beyond the interpolation threshold: error may first fall, then spike near the smallest capacity that fits the training data, and fall again for more overparameterized models.

The phenomenon is not a law that every neural experiment must display. Its shape depends on sample size, label noise, optimizer, training duration, regularization, architecture, and the axis used for “capacity.” Width, parameter count, training epochs, and dataset size create different sweeps. Near interpolation, finite optimization can also make one width appear worse simply because it trains more slowly.

Related observations include benign overfitting, where an interpolating model can still achieve low population error under specific data/noise conditions, and grokking, where training accuracy saturates long before a sudden validation improvement. These phenomena are reminders that train error and nominal parameter count are incomplete state variables. They do not remove the need for held-out evaluation.

![A schematic risk curve contrasts the classical U-shape with double descent beyond the interpolation threshold.](assets/dl06-double-descent.png){fig-align="center" width="74%" fig-alt="Schematic test-risk curves showing classical bias variance trade-off and double descent around an interpolation threshold."}

*Image source: local educational redraw based on the capacity-risk discussion in [Belkin et al., Reconciling modern machine-learning practice and the classical bias-variance trade-off](https://doi.org/10.1073/pnas.1903070116).* 

<details>
<summary><strong>PyTorch: run a cautious width sweep on Digits</strong></summary>

```python
width_sweep = []
for width in (4, 16, 64, 256, 512):
    seed_everything(613)
    model = DigitsMLP(widths=(width,))
    history = fit_model(model, epochs=25, lr=3e-3, seed=613)
    parameters = sum(p.numel() for p in model.parameters())
    width_sweep.append({
        "width": width,
        "parameters": parameters,
        "train_error": 1 - history[-1]["train"]["accuracy"],
        "validation_error": 1 - history[-1]["validation"]["accuracy"],
    })

assert [row["parameters"] for row in width_sweep] == sorted(row["parameters"] for row in width_sweep)
assert all(0 <= row["validation_error"] <= 1 for row in width_sweep)
print(width_sweep)
```

</details>

This small sweep is a **measurement template**, not evidence that Digits exhibits double descent. A defensible claim would repeat seeds, train each width to comparable convergence, locate the interpolation threshold, vary label noise, and report uncertainty. When the expected curve does not appear, that is a result rather than a reason to hide the run.

### **Experimental Protocol and Hyperparameter Selection** {#experimental-protocol-hyperparameter-selection}

An experimental protocol should be written before the most attractive result is known. At minimum it fixes the data version and split logic, primary metric, preprocessing fit boundary, model-selection budget, stopping rule, and test-access policy. Without this contract, each run quietly creates another opportunity to select noise.

Hyperparameters include architecture depth/width, augmentation strength, optimizer, learning rate, weight decay, batch size, seed policy, and stopping patience. Grid search is transparent but grows exponentially. Random search allocates more trials to distinct values when only a few dimensions matter. Bayesian and bandit methods can be more sample-efficient, but adaptivity does not make the validation set infinite; the search budget remains part of the reported method.

Use nested reasoning even when full nested cross-validation is too costly:

- **inner development loop:** train on training data and rank configurations on validation data;
- **confirmation loop:** rerun the selected configuration across prespecified seeds, optionally refitting on train plus validation;
- **outer evaluation:** evaluate once on the locked test set.

For small datasets, cross-validation can reduce split sensitivity. Deep-learning comparisons often use repeated stratified splits or repeated seeds because full nested training is expensive. Whatever design is chosen, report what varies and what stays fixed.

<details>
<summary><strong>PyTorch: select weight decay without touching test labels</strong></summary>

```python
search_records = []
for decay in (0.0, 1e-4, 1e-3, 1e-2):
    seed_everything(614)
    candidate = DigitsMLP(widths=(128, 64), dropout=0.1)
    history = fit_model(candidate, epochs=25, lr=3e-3, weight_decay=decay, seed=614)
    search_records.append((history[-1]["validation"]["loss"], decay, copy.deepcopy(candidate.state_dict())))

selected_loss, selected_decay, selected_state = min(search_records, key=lambda row: row[0])
selected_model = DigitsMLP(widths=(128, 64), dropout=0.1)
selected_model.load_state_dict(selected_state)
locked_test_result = classification_metrics(selected_model, x_test, y_test)

assert selected_decay in {0.0, 1e-4, 1e-3, 1e-2}
assert 0.0 <= locked_test_result["accuracy"] <= 1.0
print({"selected_decay": selected_decay, "validation_loss": selected_loss,
       "locked_test": locked_test_result})
```

</details>

Only one test result is produced after selection. Choosing the decay that looks best on `x_test` would turn the test set into another validation set and make its reported accuracy optimistically biased.

### **Ablation Studies, Random Seeds, and Experiment Tracking** {#ablation-studies-random-seeds-experiment-tracking}

An **ablation** removes or changes one component to test a causal question about a complete system. “Our model beats a baseline” does not reveal whether the gain came from architecture, more parameters, longer training, stronger augmentation, or a larger search budget. A clean ablation keeps data, initialization policy, update count, evaluation code, and all irrelevant choices fixed.

Random seeds affect initialization, minibatch order, stochastic regularization, and sometimes nondeterministic kernels. One seed is one realization, not a confidence interval. For paired comparisons, use the same seed set for each treatment and analyze per-seed differences

$$
d_s=M_{A,s}-M_{B,s}.
$$

The mean $\bar d$ estimates the treatment difference under the chosen seed distribution; its spread reveals stability. With only three or five seeds, avoid strong distributional claims. Show every run and state whether hyperparameters were selected before or after the seed study.

Experiment tracking should capture configuration, code revision, dataset identity, split indices or hashes, software versions, hardware, random seeds, metrics over time, and checkpoint lineage. A dashboard is useful, but a machine-readable local record is sufficient if it makes the run reconstructable.

<details>
<summary><strong>PyTorch: run a paired dropout ablation across seeds</strong></summary>

```python
paired_runs = []
for seed in (615, 616, 617):
    per_seed = {"seed": seed}
    for label, dropout_rate in (("baseline", 0.0), ("dropout", 0.2)):
        seed_everything(seed)
        model = DigitsMLP(widths=(128, 64), dropout=dropout_rate)
        history = fit_model(model, epochs=25, lr=3e-3, weight_decay=1e-3, seed=seed)
        per_seed[label] = history[-1]["validation"]["accuracy"]
    per_seed["difference"] = per_seed["dropout"] - per_seed["baseline"]
    paired_runs.append(per_seed)

differences = torch.tensor([row["difference"] for row in paired_runs])
summary = {
    "mean_difference": differences.mean().item(),
    "sample_std": differences.std(unbiased=True).item(),
    "all_runs": paired_runs,
}
assert len(paired_runs) == 3
assert torch.isfinite(differences).all()
print(summary)
```

</details>

A negative mean would not make dropout universally bad; it would say that this rate did not help this architecture, dataset, budget, and seed set. That narrower statement is exactly what a controlled ablation can support.

### **Data Leakage and Reproducibility Failures** {#data-leakage-reproducibility-failures}

**Data leakage** occurs when information unavailable at the intended prediction time influences training or model selection. Common channels include fitting normalization on all rows, selecting features with test labels, augmenting before splitting, allowing the same patient or document into multiple splits, and tuning repeatedly against a public leaderboard.

The correct split unit is the independent entity implied by deployment. If one user contributes many records, row-level random splitting lets identity-specific patterns cross the boundary. Time-dependent applications usually require chronological splits because future statistics are unavailable in the past. Near-duplicate images and derived crops should remain in the same group.

Reproducibility is broader than setting a seed. Seeds cannot repair a missing dataset version, changed preprocessing, nondeterministic hardware kernel, undocumented filtering rule, or overwritten checkpoint. Distinguish:

- **repeatability:** same code, data, environment, and hardware produce a close result;
- **reproducibility:** an independent implementation supports the claimed result;
- **replicability/generalization:** the conclusion survives a new sample, site, or setting.

<details>
<summary><strong>Python: detect augmentation leakage with provenance IDs</strong></summary>

```python
# A tempting but invalid pipeline augments each sample and then randomly splits rows.
base_ids = torch.arange(len(images))
duplicated_images = torch.cat([images, translate_digits(images, seed=618)])
duplicated_targets = torch.cat([targets, targets])
provenance = torch.cat([base_ids, base_ids])

generator = torch.Generator().manual_seed(618)
row_order = torch.randperm(len(duplicated_images), generator=generator)
cut = int(0.8 * len(row_order))
bad_train_ids = set(provenance[row_order[:cut]].tolist())
bad_test_ids = set(provenance[row_order[cut:]].tolist())
cross_boundary_duplicates = bad_train_ids & bad_test_ids

# The chapter pipeline splits base IDs first and augments only inside training.
good_train_ids = set(train_idx.tolist())
good_test_ids = set(test_idx.tolist())

assert len(cross_boundary_duplicates) > 0
assert good_train_ids.isdisjoint(good_test_ids)
print({"leaked identities in bad split": len(cross_boundary_duplicates),
       "leaked identities in chapter split": len(good_train_ids & good_test_ids)})
```

</details>

The labels were never explicitly copied into test preprocessing, yet related images crossed the boundary and made memorization useful. Provenance IDs and group-aware assertions catch a class of leakage that metric inspection alone cannot.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Generalization is not inferred from a low training loss, a fashionable regularizer, or one favorable test number. It is a claim about behavior beyond observed examples, supported by a defensible information boundary and controlled evidence.

| Concept or tool | What it changes | Evidence to inspect | Characteristic failure |
|---|---|---|---|
| capacity and training time | reachable functions and degree of fitting | matched learning curves | calling parameter count alone “overfitting” |
| explicit regularization | objective, targets, or inputs | isolated validation comparison | combining several levers and losing attribution |
| implicit regularization | optimization path and parameterization | solution norms, margins, stability | assuming one universal implicit bias |
| AdamW weight decay | multiplicative parameter shrinkage | parameter groups, norms, validation curve | decaying every parameter or copying one strength blindly |
| dropout | hidden-unit co-adaptation | train/eval-mode behavior, seed variation | evaluating in training mode or removing too much signal |
| stochastic depth | residual-path depth | path survival and deep-network stability | using it outside a suitable residual structure |
| augmentation | input transformation distribution | transformed samples and slice metrics | changing the true label or augmenting before splitting |
| MixUp/CutMix | input-target composition geometry | soft-target loss, calibration, robustness | implausible mixtures or incorrect area weights |
| label smoothing | target confidence | accuracy and calibration | stacking strong soft-target methods without an ablation |
| early stopping | optimization duration | validation trajectory and restored checkpoint | selecting the final rather than best state |
| hyperparameter search | model-selection procedure | declared space, budget, and locked test | repeatedly choosing against test results |
| ablation and seeds | causal attribution and variability | paired runs and per-seed differences | presenting one lucky seed |
| leakage controls | information availability | provenance, group/time splits, fit boundaries | random row splits across related entities |

The Digits experiments form one coherent study. They first establish an immutable split, then alter capacity, norm control, stochastic regularization, augmentation, targets, stopping, and selection one factor at a time. That continuity matters: when datasets and metrics change between examples, observed differences cannot be assigned to the mechanism being taught.

The practical sequence is:

1. Define the deployment population, prediction time, independent split unit, and primary metric.
2. Freeze the test set and fit every learned preprocessing step on training data only.
3. Establish a reproducible baseline and inspect both loss and task-facing learning curves.
4. Add one regularization mechanism because its assumption matches the task, not because it is popular.
5. Select hyperparameters on validation data under a declared budget, then confirm across fixed seeds.
6. Restore the selected checkpoint and evaluate the locked test set once.
7. Preserve configuration, provenance, environment, and results so the claim can be audited.

Regularization is therefore not a bag of tricks. It is the combination of an assumption about useful functions and an experiment capable of testing that assumption. Chapter 07 keeps the same Digits problem but changes the central question from **how a model generalizes** to **how convolutional architecture represents spatial structure**.